In [ ]:
!pip install -q streamlit bitsandbytes git+https://github.com/huggingface/peft.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%writefile app.py
import streamlit as st
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import torch

# กำหนด path ที่เก็บ full model
save_directory = "/content/drive/MyDrive/kdai-llm-miniproject-66056073/Model/full_model"

try:
    # โหลด tokenizer และ model
    tokenizer = DistilBertTokenizer.from_pretrained(save_directory)
    model = DistilBertForSequenceClassification.from_pretrained(save_directory)

    model.eval()  # ตั้งค่าเป็นโหมด evaluation

    st.title('News Classification App')
    st.write("Enter the news article below:")

    # Get user input
    user_input = st.text_area("News Article")

    labels = {
        0: "World",
        1: "Sports",
        2: "Business",
        3: "Sci/Tech"
    }

    if st.button("Classify"):
        if user_input:
            # Tokenize the input
            inputs = tokenizer(user_input,
                             return_tensors='pt',
                             truncation=True,
                             max_length=512,
                             padding=True)

            # Make prediction
            with torch.no_grad():
                outputs = model(**inputs)
            predictions = torch.argmax(outputs.logits, dim=-1).item()

            # Display the result
            st.write(f'Predicted label: {labels[predictions]}')

            # แสดงความน่าจะเป็นของแต่ละ class
            probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
            st.write("\nProbabilities for each class:")
            for label_id, label_name in labels.items():
                st.write(f"{label_name}: {probabilities[0][label_id].item():.4f}")

            # แสดงกราฟความน่าจะเป็น
            prob_dict = {label_name: probabilities[0][label_id].item()
                        for label_id, label_name in labels.items()}

            st.bar_chart(prob_dict)
        else:
            st.warning("Please enter a news article.")

except Exception as e:
    st.error(f"An error occurred while loading the model: {str(e)}")

In [ ]:
!wget -q -O - ipv4.icanhazip.com

In [ ]:
!mkdir -p ~/.streamlit/
!echo "[browser]" > ~/.streamlit/config.toml
!echo "gatherUsageStats = false" >> ~/.streamlit/config.toml

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501